In [ ]:
import requests
import json
from notebookutils import mssparkutils
from pyspark.sql.functions import col
import time

# --- Configuration ---
# Kusto Details
kusto_cluster_uri = "https://trd-6uegjpfbf030eemxtw.z1.kusto.fabric.microsoft.com"
kusto_database = "MonitoringEventhouse"
kusto_table = "AlertLogs"

# Auth Details (Update with your SPN or Key Vault logic)
tenant_id = "your-tenant-id"
client_id = "your-client-id"
client_secret = "your-client-secret" 
# Recommended: Use Key Vault
# client_secret = mssparkutils.credentials.getSecret("your-kv", "your-secret")

# API Base URL
fabric_api_base = "https://api.fabric.microsoft.com/v1"

In [ ]:
def get_spn_token(scope):
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    payload = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'scope': scope
    }
    try:
        response = requests.post(url, data=payload)
        response.raise_for_status()
        return response.json().get("access_token")
    except Exception as e:
        print(f"Failed to retrieve SPN token for scope {scope}: {e}")
        return None

# 1. Get Fabric Token (for Delete API)
fabric_scope = "https://api.fabric.microsoft.com/.default" 
fabric_token = get_spn_token(fabric_scope)

# 2. Get Kusto Token (for Reading Data)
# Note: Kusto scope is typically {ClusterURI}/.default
kusto_scope = f"{kusto_cluster_uri}/.default"
kusto_token = get_spn_token(kusto_scope)

if fabric_token and kusto_token:
    print("Successfully retrieved tokens for Fabric and Kusto.")
else:
    print("Failed to retrieve one or both tokens.")

In [ ]:
# --- Define Delete Function ---
def delete_fabric_item(workspace_id, item_id, max_retries=2):
    """
    Deletes a specific item from a Fabric Workspace.
    API: DELETE https://api.fabric.microsoft.com/v1/workspaces/{workspaceId}/items/{itemId}
    """
    if not fabric_token:
        print("Skipping Delete: No Fabric token available.")
        return False

    url = f"{fabric_api_base}/workspaces/{workspace_id}/items/{item_id}"
    
    headers = {
        "Authorization": f"Bearer {fabric_token}",
        "Content-Type": "application/json"
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.delete(url, headers=headers)
            
            # 200 OK or 204 No Content are typical success codes for DELETE
            if response.status_code in [200, 204]:
                print(f"SUCCESS: Deleted item {item_id} from workspace {workspace_id}.")
                return True
            elif response.status_code == 404:
                print(f"SKIPPED: Item {item_id} not found (already deleted?).")
                return True
            elif response.status_code == 429:
                # Handle Rate Limiting
                print(f"RATE LIMIT (429) Headers: {response.headers}")
                retry_after = int(response.headers.get("Retry-After", 60))
                print(f"RATE LIMIT (429): Waiting {retry_after} seconds before retry {attempt + 1}/{max_retries}...")
                time.sleep(retry_after)
                continue
            else:
                print(f"FAILED: Could not delete item {item_id}. Status: {response.status_code}, Response: {response.text}")
                return False
                
        except Exception as e:
            print(f"EXCEPTION: Error deleting item {item_id}: {e}")
            return False
    
    print(f"FAILED: Max retries ({max_retries}) exceeded for item {item_id}.")
    return False

In [ ]:
# --- Read Data from Kusto ---
try:
    print(f"Reading from Kusto table: {kusto_table}...")
    
    # Read using Spark Kusto connector
    df_alerts = spark.read \
        .format("com.microsoft.kusto.spark.synapse.datasource") \
        .option("kustoCluster", kusto_cluster_uri) \
        .option("kustoDatabase", kusto_database) \
        .option("kustoQuery", f"{kusto_table} | where ingestion_time() > ago(1d)") \
        .option("accessToken", kusto_token) \
        .load()
    
    # Collect to driver for iteration (Assuming volume is manageable for alerts)
    # The SQL for AlertLogs has columns: WorkspaceId, data_itemId
    alerts_list = df_alerts.select("WorkspaceId", "data_itemId", "data_itemKind", "data_itemName").collect()
    
    print(f"Found {len(alerts_list)} alerts to process.")
    
except Exception as e:
    print(f"Error reading from Kusto: {e}")
    alerts_list = []

In [ ]:
# --- Main Loop ---
if alerts_list:
    print("Starting deletion process...")
    for row in alerts_list:
        ws_id = row['WorkspaceId']
        item_id = row['data_itemId']
        item_name = row['data_itemName']
        
        print(f"Processing Alert: Delete '{item_name}' ({item_id}) in Workspace {ws_id}")
        
        # Call Delete API
        delete_fabric_item(ws_id, item_id)
        
        # Optional: Sleep to avoid rate limiting
        #time.sleep(1)
        
    print("Processing complete.")
else:
    print("No alerts to process.")